# Retail Product Demand Class Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `demand_class`

## 2 · Project Overview

We classify **retail product demand** into 3 levels (Low, Medium, High) based on pricing, promotions, shelf placement, reviews, and seasonality — supporting inventory and merchandising decisions.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a product's price, discount, category, brand tier, promotion status, shelf position, reviews, and seasonality score, predict its demand class.

## 5 · Why This Project Matters

- **Inventory management** depends on accurate demand forecasting.
- **Shelf optimization** maximizes revenue per square foot.
- Understanding demand drivers supports **pricing and promotion strategy**.
- Demonstrates retail analytics with mixed feature types.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | price, discount_pct, category, brand_tier, promotion_active, shelf_position, review_rating, num_reviews, seasonality_score |
| **Target** | demand_class (3 classes: Low, Medium, High) |
| **Class balance** | ~30/40/30 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "demand_class"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: demand_class
Save dir: E:\Github\Machine-Learning-Projects\Classification\Retail Product Demand Class Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 retail products with demand classification (Low, Medium, High).

In [4]:
np.random.seed(SEED)
n = 7000
price = np.round(np.random.lognormal(2.5, 0.8, n).clip(1, 500), 2)
discount_pct = np.round(np.random.uniform(0, 50, n), 1)
category = np.random.choice(["Electronics", "Clothing", "Food", "Home", "Sports"], n,
                             p=[0.2, 0.25, 0.2, 0.2, 0.15])
brand_tier = np.random.choice(["Budget", "Mid", "Premium"], n, p=[0.35, 0.4, 0.25])
brand_num = np.where(brand_tier == "Budget", 0, np.where(brand_tier == "Mid", 1, 2))
promotion_active = np.random.binomial(1, 0.3, n)
shelf_position = np.random.choice(["Bottom", "Middle", "Eye_Level"], n, p=[0.3, 0.4, 0.3])
shelf_num = np.where(shelf_position == "Bottom", 0, np.where(shelf_position == "Middle", 1, 2))
review_rating = np.round(np.random.uniform(1.0, 5.0, n), 1)
num_reviews = np.random.poisson(50, n).clip(0, 500)
seasonality_score = np.round(np.random.uniform(0, 1, n), 2)

score = (-0.003 * price + 0.02 * discount_pct + 0.3 * promotion_active
         + 0.2 * shelf_num + 0.15 * review_rating + 0.003 * num_reviews
         + 0.3 * seasonality_score - 0.1 * brand_num
         + np.random.normal(0, 0.8, n))
demand_class = np.where(score < np.percentile(score, 30), "Low",
               np.where(score < np.percentile(score, 70), "Medium", "High"))

for level in ["Low", "Medium", "High"]:
    if (demand_class == level).sum() < 20:
        idxs = np.random.choice(n, 20, replace=False)
        demand_class[idxs[:max(0, 20-(demand_class==level).sum())]] = level

df = pd.DataFrame({"price": price, "discount_pct": discount_pct, "category": category,
                    "brand_tier": brand_tier, "promotion_active": promotion_active,
                    "shelf_position": shelf_position, "review_rating": review_rating,
                    "num_reviews": num_reviews, "seasonality_score": seasonality_score,
                    "demand_class": demand_class})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['demand_class'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (7000, 10)
Class balance:
demand_class
Medium    0.4
Low       0.3
High      0.3
Name: proportion, dtype: float64


,price,discount_pct,category,brand_tier,promotion_active,shelf_position,review_rating,num_reviews,seasonality_score,demand_class
0,18.13,0.5,Clothing,Premium,0,Middle,4.7,61,0.60,Low
1,10.91,7.0,Clothing,Budget,1,Middle,1.5,53,0.59,High
2,20.45,16.1,Home,Mid,0,Middle,3.2,58,0.31,High
3,41.20,28.9,Sports,Budget,0,Eye_Level,2.3,54,0.04,Medium
4,10.10,3.8,Electronics,Budget,0,Middle,4.5,49,0.72,Medium


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (7000, 10)

Missing values:
price                0
discount_pct         0
category             0
brand_tier           0
promotion_active     0
shelf_position       0
review_rating        0
num_reviews          0
seasonality_score    0
demand_class         0
dtype: int64

Duplicate rows: 0

Dtypes:
price                float64
discount_pct         float64
category              object
brand_tier            object
promotion_active       int32
shelf_position        object
review_rating        float64
num_reviews            int32
seasonality_score    float64
demand_class          object
dtype: object

Target 'demand_class' confirmed. Value counts:
demand_class
Medium    2800
Low       2100
High      2100
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["price", "discount_pct", "review_rating", "num_reviews", "seasonality_score"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by Demand Class", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("category")[TARGET].value_counts(normalize=True).unstack().plot(kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Demand Class by Category")
axes[0].legend(fontsize=8)
df.groupby("shelf_position")[TARGET].value_counts(normalize=True).unstack().plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Demand Class by Shelf Position")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `demand_class`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
order = ["Low", "Medium", "High"]
df[TARGET].value_counts().reindex(order).plot(kind="bar", ax=ax,
    color=["salmon", "gold", "steelblue"], edgecolor="black")
ax.set_title("Product Demand Class Distribution")
plt.tight_layout()
plt.show()
print(f"Demand class counts:\n{df[TARGET].value_counts()}")

Demand class counts:
demand_class
Medium    2800
Low       2100
High      2100
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (5600, 9), Test: (1400, 9)
Train class distribution:
demand_class
2    0.4
1    0.3
0    0.3
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `category`, `brand_tier`, `shelf_position` encoded with OrdinalEncoder. Target encoded with LabelEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~30/40/30 (Low/Medium/High).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["effective_price"] = X_train["price"] * (1 - X_train["discount_pct"] / 100)
X_test["effective_price"] = X_test["price"] * (1 - X_test["discount_pct"] / 100)

X_train["review_volume_score"] = X_train["review_rating"] * np.log1p(X_train["num_reviews"])
X_test["review_volume_score"] = X_test["review_rating"] * np.log1p(X_test["num_reviews"])

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (11): ['price', 'discount_pct', 'category', 'brand_tier', 'promotion_active', 'shelf_position', 'review_rating', 'num_reviews', 'seasonality_score', 'effective_price', 'review_volume_score']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.4686
F1       : 0.4687

              precision    recall  f1-score   support

           0       0.49      0.42      0.45       420
           1       0.51      0.47      0.49       420
           2       0.43      0.50      0.46       560

    accuracy                           0.47      1400
   macro avg       0.48      0.46      0.47      1400
weighted avg       0.47      0.47      0.47      1400



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.446429           0.477778  0.645898  0.410572   0.447275  0.446429    0.022568
GaussianNB                     0.460000           0.475198  0.640557  0.452394   0.457920  0.460000    0.020628
AdaBoostClassifier             0.485000           0.473016  0.639389  0.482164   0.497294  0.485000    0.291538
RidgeClassifierCV              0.473571           0.472421       NaN  0.474024   0.476125  0.473571    0.021250
QuadraticDiscriminantAnalysis  0.452143           0.472024  0.626889  0.439201   0.450716  0.452143    0.045958
RidgeClassifier                0.468571           0.468254       NaN  0.469049   0.470637  0.468571    0.018797
LinearSVC                      0.466429           0.466270       NaN  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="macro_f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best model: catboost
Accuracy: 0.4814
F1: 0.4792


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.4735  (1.9s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[59]	valid_0's multi_logloss: 1.01295
LightGBM F1: 0.4783  (3.5s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.4814  0.4792     0.4934  0.4814
LightGBM               0.4786  0.4783     0.4838  0.4786
CatBoost               0.4743  0.4735     0.4823  0.4743
Logistic Regression    0.4686  0.4687     0.4731  0.4686

Top 3 models: ['FLAML', 'LightGBM', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='weighted'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='weighted', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.4814
    F1       : 0.4792
    Precision: 0.4934
    Recall   : 0.4814

  LightGBM:
    Accuracy : 0.4786
    F1       : 0.4783
    Precision: 0.4838
    Recall   : 0.4786

  CatBoost:
    Accuracy : 0.4743
    F1       : 0.4735
    Precision: 0.4823
    Recall   : 0.4743


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.39      0.45       420
           1       0.53      0.44      0.48       420
           2       0.44      0.58      0.50       560

    accuracy                           0.48      1400
   macro avg       0.50      0.47      0.48      1400
weighted avg       0.49      0.48      0.48      1400


Total errors: 726 / 1400 (51.9%)

Sample misclassifications:
      price  discount_pct  category  brand_tier  promotion_active  shelf_position  review_rating  num_reviews  seasonality_score  effective_price  review_volume_score  actual  predicted  correct
391   18.06          36.6       2.0         1.0                 0             1.0            3.0           44               0.52         11.45004            11.419987       2          0    False
2546   4.05          14.8       0.0         1.0                 0             1.0            2.1           49               0

## 25 · Interpretation and Business Insight

**Key findings:**
- **Promotion** and **shelf position (eye level)** are the strongest demand drivers.
- **Lower effective price** (after discount) increases demand.
- **Review rating** and **review volume** together predict demand.
- **Seasonality** adds significant signal.
- **Premium brands** paradoxically show lower demand volume (niche markets).

**Business takeaway:** Optimize shelf placement to eye level, time promotions with seasonality peaks, and monitor review quality.

## 26 · Limitations

1. Synthetic data with simplified retail dynamics.
2. No competitor pricing or substitutes.
3. Missing store location and foot traffic.
4. No temporal demand patterns.
5. Three demand classes is a simplification.

## 27 · How to Improve This Project

1. Use real POS/sales data.
2. Add competitor products and prices.
3. Include store-level features.
4. Model demand as continuous (regression).
5. Add weather and economic indicators.

## 28 · Production Considerations

- Deploy for automated inventory replenishment.
- Update weekly with new sales data.
- Integrate with pricing optimization.
- Monitor by category for demand shifts.
- Combine with supply chain lead times.

## 29 · Common Mistakes

1. Ignoring seasonality in demand predictions.
2. Not considering price elasticity by category.
3. Using accuracy without per-class analysis.
4. Treating demand classes as independent (they're ordinal).
5. Not accounting for stockouts biasing historical demand data.

## 30 · Mini Challenge / Exercises

1. Remove `promotion_active` — how much does High demand recall drop?
2. Try ordinal regression (Low < Medium < High).
3. Analyze price elasticity by category (interaction).
4. Create `value_score = review_rating / price` and test.
5. Split by category and build category-specific models.

## 31 · Final Summary / Key Takeaways

1. **Promotions** and **shelf position** are the strongest demand levers.
2. **Effective price** (after discount) drives volume.
3. **Review quality × volume** indicates product appeal.
4. **Seasonality** adds actionable timing signal.
5. **Ordinal classification** fits the natural demand ordering.
6. **Real-world demand** forecasting requires temporal and competitive data.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Retail Product Demand Class Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.4743,
    "f1": 0.4735,
    "precision": 0.4823,
    "recall": 0.4743
  },
  "LightGBM": {
    "accuracy": 0.4786,
    "f1": 0.4783,
    "precision": 0.4838,
    "recall": 0.4786
  },
  "Logistic Regression": {
    "accuracy": 0.4686,
    "f1": 0.4687,
    "precision": 0.4731,
    "recall": 0.4686
  },
  "FLAML": {
    "accuracy": 0.4814,
    "f1": 0.4792,
    "precision": 0.4934,
    "recall": 0.4814
  }
}
